Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision import datasets, transforms, models

from helper import make_dataset, train, test, get_indices

Use device

In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cpu


Fill in respective folder location for dataset

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),  
    transforms.RandomRotation(5),      
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_real_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\train\real"
train_fake_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\train\fake"
valid_real_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\train\real"
valid_fake_dir = r"C:\SUTD\50.039 Deep Learning\PROJECT\datasetrealfake\DFWILD\valid\fake"

train_dir = r"D:\archive (1)\DFWILD\train"
valid_dir = r"D:\archive (1)\DFWILD\valid"

Create train, valid and test datasets and dataloaders

In [4]:

train_dataset = make_dataset(train_dir, 9000, 1000, transform)

valid_dataset = make_dataset(valid_dir, 900, 100, transform)

val_size = int(0.5 * len(valid_dataset))
test_size = len(valid_dataset) - val_size

val_dataset, test_dataset = random_split(valid_dataset, [val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

real_count = 0
fake_count = 0

for _, labels in train_loader:
    real_count += (labels == 0).sum().item()
    fake_count += (labels == 1).sum().item()

print("Train real:", real_count)
print("Train fake:", fake_count)



Train real: 9000
Train fake: 1000


ResNet18

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Sequential(
    nn.Linear(512, 128),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(128, 32),
    nn.ReLU(inplace=True),
    nn.Dropout(0.4)
    nn.Linear(32, 2),
)
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True

In [ ]:
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0,15.0]).to(device))
optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5, 'weight_decay': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-4, 'weight_decay': 1e-4}
])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)
train(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=20, scheduler=scheduler)
test(model, test_loader, device)

Epoch : 0 
Train Loss     : 0.6864
Validation Accuracy  : 92.00%
F1 Score       : 0.0476
Recall         : 0.0244  ← how many anomalies caught
Precision      : 1.0000
AUC-ROC        : 0.5807

Confusion Matrix:
                 Predicted Real  Predicted Fake
Actual Real      459             0
Actual Fake      40              1
Epoch : 1 
Train Loss     : 0.6762
Validation Accuracy  : 91.60%
F1 Score       : 0.1600
Recall         : 0.0976  ← how many anomalies caught
Precision      : 0.4444
AUC-ROC        : 0.6233

Confusion Matrix:
                 Predicted Real  Predicted Fake
Actual Real      454             5
Actual Fake      37              4
Epoch : 2 
Train Loss     : 0.6660
Validation Accuracy  : 87.40%
F1 Score       : 0.2921
Recall         : 0.3171  ← how many anomalies caught
Precision      : 0.2708
AUC-ROC        : 0.6411

Confusion Matrix:
                 Predicted Real  Predicted Fake
Actual Real      424             35
Actual Fake      28              13
Epoch : 3 
Train 